<a href="https://colab.research.google.com/github/OMGItsYutoo/MVPNet/blob/main/DatasetBuilder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install segmentation-models-pytorch tqdm xlwt
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content')

!rm -rf MVPNet
!git clone https://github.com/OMGItsYutoo/MVPNet

os.chdir('/content/MVPNet/ml_depth_pro')

!pip install -e .

os.chdir('/content')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'MVPNet'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 153 (delta 3), reused 8 (delta 3), pack-reused 145 (from 1)
Receiving objects: 100% (153/153), 115.06 MiB | 32.02 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (146/146), done.
Obtaining file:///content/MVPNet/ml-depth-pro
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for depth_pro (pyproject.toml) ... done
  Created wheel for depth_pro: filename=depth_pro-0.1-0.editable-py3-none-any.whl size=4843 sha256=f7fe607634947dff29de19ea0a9ca9b522005dcec79e6ff29f8b91782630ea13
  Stored in director

In [ ]:
import os
os.chdir('/content/MVPNet')

!rm -rf ./ICCV2019_MirrorNet/images/image
!rm -rf ./ICCV2019_MirrorNet/images/result
!rm -rf ./ml_depth_pro/checkpoints
!rm -rf ./ml_depth_pro/data

!mkdir -p ./ICCV2019_MirrorNet/ckpt/MirrorNet/
!ln -s /content/drive/MyDrive/mirrornet/160.pth ./ICCV2019_MirrorNet/ckpt/MirrorNet/160.pth
!ln -s /content/drive/MyDrive/mirrornet/resnext_101_32x4d.pth ./ICCV2019_MirrorNet/backbone/resnext/resnext_101_32x4d.pth

!ln -s /content/drive/MyDrive/ml-depth_pro/data/image ./ICCV2019_MirrorNet/images/image
!ln -s /content/drive/MyDrive/ml-depth_pro/data/mask_sloppy ./ICCV2019_MirrorNet/images/result

!ln -s /content/drive/MyDrive/ml-depth_pro/checkpoints ./ml_depth_pro/checkpoints
!ln -s /content/drive/MyDrive/ml-depth_pro/data ./ml_depth_pro/data

print("Symlinks cleaned and recreated perfectly!")

Symlinks cleaned and recreated perfectly!


In [ ]:
!pip uninstall -y cython
!pip install "cython<3.0.0" wheel setuptools
!pip install --no-build-isolation git+https://github.com/lucasb-eyer/pydensecrf.git

Found existing installation: Cython 3.0.12
Uninstalling Cython-3.0.12:
  Successfully uninstalled Cython-3.0.12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.4 MB/s eta 0:00:00
  Cloning https://github.com/lucasb-eyer/pydensecrf.git to /tmp/pip-req-build-hw9ar5do
  Running command git clone --filter=blob:none --quiet https://github.com/lucasb-eyer/pydensecrf.git /tmp/pip-req-build-hw9ar5do
  Resolved https://github.com/lucasb-eyer/pydensecrf.git to commit 2723c7fa4f2ead16ae1ce3d8afe977724bb8f87f
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pydensecrf: filename=pydensecrf-1.0-cp312-cp312-linux_x86_64.whl size=3494358 sha256=484c331db7fbb3d2c20fd378dd2f783e363b5bb12f2a722045321f1c9daf7068
  Stored in directory: /tmp/pip-ephem-wheel-cache-igta3e1z/wheels/5f/63/9b/ad8357747651277615ec2094c768471e8ecde7d7b53564f24d
Successfully built pydensecrf


In [ ]:
import os
import glob

MIRRORNET_DIR = "/content/MVPNet/ICCV2019_MirrorNet"
DEPTHPRO_DIR = "/content/MVPNet/ml_depth_pro"

# 1. Run MirrorNet on all images
print("--- [1/2] Running MirrorNet Inference ---")
os.chdir(MIRRORNET_DIR)
!python infer.py

print("--- [2/2] Running Depth Pro & RANSAC ---")
os.chdir(DEPTHPRO_DIR)

image_files = glob.glob('./data/image/*.jpg')

for filepath in image_files:
    filename = os.path.basename(filepath)

    suffix = filename.replace('img', '').replace('.jpg', '')

    print(f"\n==========================================")
    print(f"Processing Image Suffix: {suffix}")
    print(f"==========================================")

    !python infer_ransac_gt.py --img {suffix}

print("\n--- Full Pipeline Complete! ---")

Output streaming troncato alle ultime 5000 righe.
Preparing final plots...

Saving predicted depth maps...
Figure(1800x600)

Processing Image Suffix: _832
Loading Apple Depth Pro...
Loading image: ./data/image/img_832.jpg
Running Apple Depth Pro inference...
Processing Predicted Depth with perfect Mask
  -> Loading sloppy mask: ./data/mask/img_832.png
  -> Applying RANSAC geometric correction...

Processing Predicted Depth with SLOPPY Mask...
  -> Loading sloppy mask: ./data/mask_sloppy/img_832.png
  -> Applying RANSAC geometric correction...

Preparing final plots...

Saving predicted depth maps...
Figure(1800x600)

Processing Image Suffix: _950
Loading Apple Depth Pro...
Loading image: ./data/image/img_950.jpg
Running Apple Depth Pro inference...
Processing Predicted Depth with perfect Mask
  -> Loading sloppy mask: ./data/mask/img_950.png
  -> Applying RANSAC geometric correction...

Processing Predicted Depth with SLOPPY Mask...
  -> Loading sloppy mask: ./data/mask_sloppy/img_950.

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import numpy as np
import cv2
from IPython.display import display

DEPTHPRO_DIR = "/content/MVPNet/ml_depth_pro"
os.chdir(DEPTHPRO_DIR)

def get_inverse_depth_viz(depth_matrix):
    """Normalizza la depth per la visualizzazione (Turbo colormap)"""
    inv_depth = 1 / depth_matrix
    max_inv = min(inv_depth.max(), 1 / 0.1)
    min_inv = max(1 / 250, inv_depth.min())
    return (inv_depth - min_inv) / (max_inv - min_inv)

processed_files = sorted(glob.glob('./data/depth_corr/*.npy'))

files_to_show = processed_files[0:31]

print(f"Trovati {len(processed_files)} risultati totali. Visualizzazione di {len(files_to_show)} immagini...\n")

for corr_path in files_to_show:
    filename = os.path.basename(corr_path)
    base_name = filename.replace('.npy', '')

    img_path = f"./data/image/{base_name}.jpg"
    mask_sloppy_path = f"./data/mask_sloppy/{base_name}.png"
    mask_clean_path = f"./data/mask/{base_name}.png"
    depth_orig_path = f"./data/depth/{base_name}.npy"
    depth_gt_path = f"./data/depth_gt/{base_name}.npy"

    if not os.path.exists(img_path) or not os.path.exists(depth_orig_path):
        continue

    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    depth_orig = np.load(depth_orig_path)
    depth_corr_sloppy = np.load(corr_path)

    mask_sloppy = cv2.imread(mask_sloppy_path, cv2.IMREAD_GRAYSCALE) if os.path.exists(mask_sloppy_path) else np.zeros_like(img[:,:,0])
    mask_clean = cv2.imread(mask_clean_path, cv2.IMREAD_GRAYSCALE) if os.path.exists(mask_clean_path) else np.zeros_like(img[:,:,0])

    if os.path.exists(depth_gt_path):
        depth_gt = np.load(depth_gt_path)
    else:
        print(f"GT mancante per {base_name}, mostro la depth originale.")
        depth_gt = depth_orig.copy()

    inv_orig = get_inverse_depth_viz(depth_orig)
    inv_sloppy = get_inverse_depth_viz(depth_corr_sloppy)
    inv_gt = get_inverse_depth_viz(depth_gt)

    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Confronto Risultati per: {base_name}", fontsize=18, fontweight='bold')

    axs[0, 0].imshow(img)
    axs[0, 0].set_title("Immagine Originale")
    axs[0, 0].axis("off")

    axs[0, 1].imshow(mask_sloppy, cmap="gray")
    axs[0, 1].set_title("Maschera Sloppy (MirrorNet)")
    axs[0, 1].axis("off")

    axs[0, 2].imshow(inv_sloppy, cmap="turbo")
    axs[0, 2].set_title("Depth Corretta (Sloppy pre-salvata)")
    axs[0, 2].axis("off")

    axs[1, 0].imshow(inv_orig, cmap="turbo")
    axs[1, 0].set_title("Depth Pro Originale")
    axs[1, 0].axis("off")

    axs[1, 1].imshow(mask_clean, cmap="gray")
    axs[1, 1].set_title("Maschera Pulita")
    axs[1, 1].axis("off")

    axs[1, 2].imshow(inv_gt, cmap="turbo")
    axs[1, 2].set_title("Depth GT (Clean pre-salvata)")
    axs[1, 2].axis("off")

    plt.tight_layout()
    plt.show()